# Sesión 5 · La muestra que se eligió sola
**Análisis Estadístico de la Información — Especialización en Big Data (UCompensar)**

Este cuaderno (*notebook*) acompaña las [diapositivas de la sesión](https://jotamao1985.github.io/ANaLISIS-ESTADiSTICO-DE-LA-INFORMACIoN_Ucomp/diapositivas/sesion-5-la-muestra-que-se-eligio-sola.html).
Cada celda lleva entre corchetes el mismo identificador que el código proyectado, así que puedes seguir
la clase celda a celda. La portada del curso está en https://jotamao1985.github.io/ANaLISIS-ESTADiSTICO-DE-LA-INFORMACIoN_Ucomp/
y el enunciado del proyecto final, el tablero, en
[su página](https://jotamao1985.github.io/ANaLISIS-ESTADiSTICO-DE-LA-INFORMACIoN_Ucomp/proyecto-tablero.html).

1. **Archivo → Guardar una copia en Drive** para que tus cambios no se pierdan.
2. Ejecuta las celdas **en orden** con **Mayús + Enter**. Varias usan resultados de las anteriores.
   Si Colab avisa que el cuaderno no es de Google, elige **Ejecutar de todos modos**.
3. Si Colab se reinicia o una celda falla al repetirla, usa **Entorno de ejecución → Ejecutar anteriores**.
4. La primera celda crea los datos del caso *Tingua Domicilios S.A.S.*, una empresa **ficticia**
   (sus datos se inventan con una semilla fija y no corresponden a personas ni a empresas reales),
   y les da a las gráficas los colores del curso.

In [ ]:
#@title ▶ Crea los datos del caso (ejecútala sin modificar) { display-mode: "form" }
"""Datos del caso «Tingua Domicilios S.A.S.» (empresa FICTICIA).

Genera datos/pedidos.csv: los 12 000 pedidos que la plataforma entregó entre el
1 de enero y el 30 de junio de 2026 en cinco ciudades. Todo se inventa con una
semilla fija: los datos no corresponden a personas ni a empresas reales.

Solo usa la biblioteca estándar y, del generador aleatorio, solo random(), que
es lo único que Python garantiza igual entre versiones. Por eso la tabla sale
idéntica en Colab y en cualquier computador.

El caso sirve para todo el curso: variables de todos los tipos (sesión 1),
asimetrías y algunos atípicos (medidas y datos atípicos), una población
completa de la que se pueden sacar muestras (muestreo) y un tiempo de entrega
que depende de la distancia, la lluvia y otras variables (regresión).
"""

import csv
import math
import random
from datetime import date, timedelta
from pathlib import Path

SEMILLA = 2026
N_PEDIDOS = 12_000
INICIO = date(2026, 1, 1)
DIAS = 181  # del 1 de enero al 30 de junio de 2026

COLUMNAS = ["id_pedido", "fecha", "franja", "ciudad", "categoria", "n_productos",
            "valor_pedido", "medio_pago", "estrato", "suscriptor", "distancia_km",
            "lluvia", "temperatura_c", "tiempo_entrega_min", "calificacion"]

# ciudad: (peso, temperatura media °C, desviación, prob. de lluvia, minutos extra, factor de distancia)
CIUDADES = {
    "Bogotá":       (0.40, 14.5, 2.2, 0.30, 5.0, 1.15),
    "Medellín":     (0.22, 22.5, 2.3, 0.28, 2.0, 1.00),
    "Cali":         (0.16, 25.0, 2.3, 0.22, 0.0, 1.00),
    "Barranquilla": (0.12, 29.0, 1.8, 0.12, -1.0, 0.95),
    "Bucaramanga":  (0.10, 23.5, 2.2, 0.20, -2.0, 0.90),
}
LLUVIA_MES = {1: 0.6, 2: 0.7, 3: 1.0, 4: 1.4, 5: 1.3, 6: 0.8}

# franja: (peso, ajuste de temperatura °C, minutos extra)
FRANJAS = {
    "Mañana":   (0.15, -1.5, 0.0),
    "Almuerzo": (0.35, 1.5, 4.0),
    "Tarde":    (0.20, 1.0, 0.0),
    "Noche":    (0.30, -2.0, 3.0),
}

# categoría: (productos extra en promedio, precio medio por producto en COP, minutos de preparación)
CATEGORIAS = {
    "Restaurante": (1.3, 27_000, 3.0),
    "Mercado":     (6.0, 8_500, 4.0),
    "Farmacia":    (0.9, 21_000, 0.0),
    "Mascotas":    (1.2, 36_000, 1.0),
    "Floristería": (0.2, 88_000, 2.0),
}
# Qué se pide en cada franja (mismo orden que CATEGORIAS)
CATEGORIA_POR_FRANJA = {
    "Mañana":   (0.30, 0.35, 0.20, 0.10, 0.05),
    "Almuerzo": (0.78, 0.08, 0.07, 0.04, 0.03),
    "Tarde":    (0.40, 0.25, 0.15, 0.12, 0.08),
    "Noche":    (0.70, 0.10, 0.12, 0.05, 0.03),
}
ESTRATOS = {1: 0.06, 2: 0.22, 3: 0.33, 4: 0.20, 5: 0.11, 6: 0.08}


class _Azar:
    """Lo que hace falta del azar, construido solo sobre random()."""

    def __init__(self, semilla):
        self.r = random.Random(semilla)

    def u(self):
        return self.r.random()

    def elegir(self, opciones, pesos):
        x, acum = self.u() * sum(pesos), 0.0
        for opcion, peso in zip(opciones, pesos):
            acum += peso
            if x < acum:
                return opcion
        return opciones[-1]

    def normal(self, media=0.0, desv=1.0):
        # Box-Muller
        u1, u2 = 1.0 - self.u(), self.u()
        return media + desv * math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)

    def exponencial(self, media):
        return -media * math.log(1.0 - self.u())

    def poisson(self, lam):
        # Knuth: suficiente para medias pequeñas
        limite, k, p = math.exp(-lam), 0, 1.0
        while True:
            p *= self.u()
            if p <= limite:
                return k
            k += 1


def _redondear(x, decimales=0):
    factor = 10 ** decimales
    return math.floor(x * factor + 0.5) / factor


def _pedido(az, dias, pesos_dias):
    fecha = az.elegir(dias, pesos_dias)
    ciudad = az.elegir(list(CIUDADES), [v[0] for v in CIUDADES.values()])
    _, t_media, t_desv, p_lluvia, min_ciudad, f_dist = CIUDADES[ciudad]
    franja = az.elegir(list(FRANJAS), [v[0] for v in FRANJAS.values()])
    _, t_franja, min_franja = FRANJAS[franja]
    categoria = az.elegir(list(CATEGORIAS), CATEGORIA_POR_FRANJA[franja])
    extra, precio, min_prep = CATEGORIAS[categoria]
    estrato = az.elegir(list(ESTRATOS), list(ESTRATOS.values()))

    suscriptor = az.u() < 0.10 + 0.06 * (estrato - 1)
    p_efectivo = 0.45 - 0.07 * (estrato - 1)
    medio_pago = az.elegir(["Efectivo", "Billetera digital", "Tarjeta"],
                           [p_efectivo, 0.25, 1 - p_efectivo - 0.25])

    n = 1 + az.poisson(extra)
    if categoria in ("Mercado", "Restaurante") and az.u() < 0.003:
        n = 20 + int(az.u() * 26)  # pedido corporativo: pocos, pero existen
    valor = n * precio * math.exp(az.normal(0, 0.22)) * (1 + 0.07 * (estrato - 3))
    valor = max(3_000, _redondear(valor / 100) * 100)

    distancia = math.exp(az.normal(math.log(3.0), 0.5)) * f_dist
    distancia = min(14.0, max(0.4, _redondear(distancia, 1)))

    lluvia = az.u() < p_lluvia * LLUVIA_MES[fecha.month]
    temperatura = _redondear(az.normal(t_media + t_franja - 1.8 * lluvia, t_desv), 1)

    tiempo = (11 + 3.1 * distancia + 0.45 * min(n, 15) + min_prep + min_franja + min_ciudad
              + 7.5 * lluvia - 2.5 * suscriptor + az.normal(0, 4) + az.exponencial(3))
    if az.u() < 0.005:
        tiempo += 55 + 95 * az.u()  # pedidos que se demoraron muchísimo
    tiempo = max(6.0, _redondear(tiempo, 1))

    latente = 6.4 - 0.065 * tiempo + 0.2 * suscriptor + az.normal(0, 0.6)
    estrellas = int(min(5, max(1, math.floor(latente + 0.5))))
    califico = az.u() < (0.80 if tiempo > 55 else 0.58)

    return {
        "fecha": fecha, "franja": franja, "ciudad": ciudad, "categoria": categoria,
        "n_productos": n, "valor_pedido": int(valor), "medio_pago": medio_pago,
        "estrato": estrato, "suscriptor": "Sí" if suscriptor else "No",
        "distancia_km": distancia, "lluvia": "Sí" if lluvia else "No",
        "temperatura_c": temperatura, "tiempo_entrega_min": tiempo,
        "calificacion": estrellas if califico else "",
    }


def crear_datos(carpeta="datos"):
    """Escribe <carpeta>/pedidos.csv y devuelve su ruta."""
    az = _Azar(SEMILLA)
    dias = [INICIO + timedelta(days=d) for d in range(DIAS)]
    pesos_dias = [1.35 if d.weekday() >= 4 else 1.0 for d in dias]  # viernes a domingo, más pedidos
    pedidos = [_pedido(az, dias, pesos_dias) for _ in range(N_PEDIDOS)]
    orden_franja = list(FRANJAS)
    pedidos.sort(key=lambda p: (p["fecha"], orden_franja.index(p["franja"])))

    ruta = Path(carpeta) / "pedidos.csv"
    ruta.parent.mkdir(parents=True, exist_ok=True)
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        escritor = csv.DictWriter(f, fieldnames=COLUMNAS)
        escritor.writeheader()
        for i, p in enumerate(pedidos, start=1):
            escritor.writerow({"id_pedido": f"P{i:05d}", **p, "fecha": p["fecha"].isoformat()})
    miles = f"{N_PEDIDOS:,}".replace(",", "\u202f")  # 12 000, con espacio fino
    print(f"Listo: {ruta} con {miles} pedidos y {len(COLUMNAS)} columnas")
    return ruta


"""Ayudas comunes a todos los notebooks del curso (van en la celda oculta, tras genera_datos.py).

Las de cada sesión están en ayudas_sesionN.py.
"""


def _aviso_version():
    """Las diapositivas se ejecutaron con pandas 2.2; con pandas 3 algunas salidas cambian."""
    import pandas as pd

    if int(pd.__version__.split(".")[0]) >= 3:
        print(f"Aviso: este Colab trae pandas {pd.__version__} y las diapositivas usan pandas 2.2. "
              "Algunas salidas se verán un poco distintas (por ejemplo, el texto sale como str "
              "y no como object).")


_aviso_version()


"""Sesión 5: estilo de las gráficas del curso, la calificación completa y los verificadores.

Van dentro de la celda oculta del notebook, junto con genera_datos.py y ayudas.py. Cada
verificador recibe lo que el estudiante calculó, dice qué está bien y, en lo demás, da una
pista sin la respuesta. Nunca se cae: un «...» sin reemplazar o un tipo inesperado se avisa.

calificacion_completa() repite el generador de la celda oculta para recuperar la calificación
que habría dado cada pedido. Solo existe porque el caso es ficticio: se usa una sola vez, en la
celda [comprobacion], y nunca en el tablero.

Tolerancias: ±1 pedido por ciudad en el ejercicio 1 (cada ciudad redondea), ±0,06 puntos
porcentuales en el ejercicio 2 (una proporción entre 0 y 1 se acepta con una pista) y conteos
exactos en el ejercicio 3, que no supone cuántos pedidos trae la muestra.
"""


def _estilo_graficas():
    """Colores del curso y letras que se leen en Colab y proyectadas."""
    import matplotlib as mpl
    import matplotlib.pyplot as plt

    plt.rcParams.update({
        "axes.prop_cycle": mpl.cycler(color=["#8A2E9E", "#F46201", "#566B7F"]),
        "figure.figsize": (6.4, 3.6),
        "figure.constrained_layout.use": True,
        "font.size": 13,
        "axes.titlesize": 15,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#94A3B8",
        "axes.labelcolor": "#334155",
        "xtick.color": "#475569",
        "ytick.color": "#475569",
        "text.color": "#1E293B",
    })


_estilo_graficas()


def _mostrar(tabla):
    try:
        from IPython.display import display
        display(tabla)
    except ImportError:
        print(tabla.to_string())


def _original():
    """Los pedidos tal como salen del generador, o None (con aviso) si no están."""
    import pandas as pd

    try:
        return pd.read_csv("datos/pedidos.csv")
    except FileNotFoundError:
        print("✗  No encuentro datos/pedidos.csv: ejecuta primero la celda que crea los datos "
              "(Entorno de ejecución → Ejecutar anteriores)")
        return None


def _sobrantes(sobran, otros, nombres):
    """Avisa de argumentos de más o con otro nombre. True si hubo alguno."""
    if otros:
        print(f"✗  No conozco {', '.join(repr(k) for k in otros)}: usa estos nombres, en "
              f"minúsculas y sin tildes: {', '.join(n + '=' for n in nombres)}")
        return True
    if sobran:
        print(f"✗  Sobran valores: el verificador recibe solo {len(nombres)} "
              f"({', '.join(n + '=' for n in nombres)})")
        return True
    return False


def _seguro(verificador):
    """Red de seguridad: si algo imprevisto falla al revisar, se avisa sin traceback."""
    import functools

    @functools.wraps(verificador)
    def envoltura(*args, **kwargs):
        try:
            return verificador(*args, **kwargs)
        except Exception as error:   # noqa: BLE001 — el estudiante nunca ve un traceback
            print(f"✗  No pude revisar lo que entregaste ({type(error).__name__}). Revisa que lo que "
                  "entregas sea el resultado de un cálculo de pandas y vuelve a ejecutar la celda")
    return envoltura


_COLUMNAS = {"id_pedido", "fecha", "franja", "ciudad", "categoria", "n_productos", "valor_pedido",
             "medio_pago", "estrato", "suscriptor", "distancia_km", "lluvia", "temperatura_c",
             "tiempo_entrega_min", "calificacion"}


def _un_numero(x, n_columna=None):
    """(valor, None) si x es un número, o una serie o lista con un solo número; si no, (None, motivo)."""
    import math
    import numbers
    import numpy as np
    import pandas as pd

    if isinstance(x, pd.DataFrame):
        if x.size != 1:
            return None, f"es una tabla de {x.shape[0]} × {x.shape[1]}; hace falta un solo número"
        x = x.iloc[0, 0]
    if isinstance(x, (pd.Series, pd.Index, np.ndarray, list, tuple)):
        valores = list(np.ravel(np.asarray(x, dtype=object)))
        if n_columna is not None and len(valores) == n_columna:
            return None, "esa es la columna entera, un valor por pedido: falta resumirla"
        if len(valores) == 0:
            return None, "está vacío"
        if len(valores) > 1:
            if all(isinstance(v, float) and math.isnan(v) for v in valores):
                return None, ("salió nan: si restaste dos series, pandas las alinea por su índice "
                              "y lo que no coincide queda nan; calcula cada valor como un número")
            pista = ""
            if isinstance(x, pd.Series) and set(map(str, x.index)) & _COLUMNAS:
                pista = ": ¿elegiste la columna?"   # p. ej., pedidos.quantile(0.25, numeric_only=True)
            elif isinstance(x, pd.Series) and {"count", "mean", "std"} <= set(map(str, x.index)):
                pista = ": describe() da varias medidas; quédate solo con la que se pide"
            return None, f"tiene {len(valores)} valores; hace falta uno solo{pista}"
        x = valores[0]   # quantile([0.25]) devuelve una serie de un solo valor
    if x is None:
        return None, "está vacío (None): ¿le asignaste el resultado de un print?"
    if x is pd.NA:
        return None, "salió <NA>: revisa el cálculo (¿la columna tiene datos?)"
    if callable(x):
        return None, "es una función, no su resultado: ¿faltaron los paréntesis del final, ()?"
    if isinstance(x, (bool, np.bool_)):
        return None, "es True o False; hace falta un número"
    if isinstance(x, str):
        return None, "es texto; escríbelo como número, sin comillas y con punto decimal"
    if not isinstance(x, (numbers.Real, np.number)) or isinstance(x, np.complexfloating):
        return None, f"no es un número real (es {type(x).__name__})"
    try:
        valor = float(x)
    except (TypeError, ValueError, OverflowError):
        return None, "no es un número real"
    if math.isnan(valor):
        return None, "salió nan: revisa el cálculo (¿la columna tiene datos?)"
    if math.isinf(valor):
        return None, "salió infinito: ¿dividiste entre cero?"
    return valor, None


def _cerca(a, b, tolerancia):
    return a is not None and abs(a - b) <= tolerancia


def _relativo(a, b, tolerancia):
    """¿a está a menos de `tolerancia` (relativa) de b? Para pesos."""
    return a is not None and abs(a - b) <= tolerancia * max(abs(b), 1.0)


# ---------------------------------------------------------------------------------------------
# La calificación completa (solo para la comprobación de la sesión 5)
# ---------------------------------------------------------------------------------------------

def calificacion_completa():
    """La calificación que habría dado cada pedido si todos calificaran. Solo en un caso ficticio.

    Repite el generador de la celda oculta grabando los números al azar de cada pedido y vuelve a
    correr cada pedido con esa misma cinta, pero con el último número (el que decide si califica)
    puesto en 0: así todos califican y lo demás no cambia. Devuelve una serie de enteros indexada
    por id_pedido, que se cruza con map, por código. No escribe archivos. Si algo no cuadra con
    datos/pedidos.csv, avisa y devuelve None.
    """
    import numpy as np
    import pandas as pd
    from datetime import timedelta

    try:
        base = _Azar
        generar = _pedido
        semilla, total, inicio, n_dias, orden_franjas = SEMILLA, N_PEDIDOS, INICIO, DIAS, list(FRANJAS)
    except NameError:
        print("✗  No encuentro el generador de los datos: ejecuta primero la celda que crea los datos "
              "(Entorno de ejecución → Ejecutar anteriores)")
        return None

    class _Graba(base):
        """El azar de siempre, pero anota cada número que entrega, pedido por pedido."""

        def __init__(self, s):
            super().__init__(s)
            self.cinta = []

        def u(self):
            x = super().u()
            self.cinta[-1].append(x)
            return x

    class _Repite(base):
        """Entrega los números de una cinta grabada, en el mismo orden."""

        def __init__(self, cinta):
            self.cinta, self.i = cinta, 0

        def u(self):
            x = self.cinta[self.i]
            self.i += 1
            return x

    try:
        original = _original()
        if original is None:
            return None
        dias = [inicio + timedelta(days=d) for d in range(n_dias)]
        pesos_dias = [1.35 if d.weekday() >= 4 else 1.0 for d in dias]
        az = _Graba(semilla)
        vistos, completos = [], []
        for _ in range(total):
            az.cinta.append([])
            vistos.append(generar(az, dias, pesos_dias))
            cinta = az.cinta[-1][:-1] + [0.0]       # el último número decide si califica
            repite = _Repite(cinta)
            completos.append(generar(repite, dias, pesos_dias))
            if repite.i != len(cinta):
                raise ValueError("la cinta no se consumió entera")
        orden = sorted(range(total), key=lambda i: (vistos[i]["fecha"],
                                                    orden_franjas.index(vistos[i]["franja"])))
        ids = [f"P{k:05d}" for k in range(1, total + 1)]
        # 1) Las demás columnas no cambian al hacer que todos califiquen
        for i in orden:
            a, b = vistos[i], completos[i]
            if any(a[c] != b[c] for c in a if c != "calificacion") or b["calificacion"] == "":
                raise ValueError("cambió otra columna")
        # 2) Coincide con datos/pedidos.csv: los mismos pedidos y las calificaciones visibles
        tabla = pd.DataFrame([vistos[i] for i in orden])
        tabla["fecha"] = [vistos[i]["fecha"].isoformat() for i in orden]
        if (len(original) != total or list(original["id_pedido"].astype(str)) != ids
                or list(original["fecha"].astype(str)) != list(tabla["fecha"])):
            raise ValueError("los pedidos no son los del archivo")
        for c in ["franja", "ciudad", "categoria", "medio_pago", "suscriptor", "lluvia"]:
            if list(original[c].astype(str)) != list(tabla[c].astype(str)):
                raise ValueError(f"cambió {c}")
        for c in ["n_productos", "valor_pedido", "estrato", "distancia_km", "temperatura_c",
                  "tiempo_entrega_min"]:
            if not np.allclose(original[c].to_numpy(dtype=float), tabla[c].to_numpy(dtype=float),
                               rtol=0, atol=1e-9):
                raise ValueError(f"cambió {c}")
        if list(original["calificacion"].isna()) != [v == "" for v in tabla["calificacion"]]:
            raise ValueError("cambió qué pedidos calificaron")
        completa = pd.Series([int(completos[i]["calificacion"]) for i in orden],
                             index=pd.Index(ids, name="id_pedido"), name="calificacion_completa")
        visibles = original.set_index("id_pedido")["calificacion"].dropna()
        if not (completa.loc[visibles.index] == visibles.astype(int)).all():
            raise ValueError("no coincide con las calificaciones visibles")
        return completa
    except Exception as error:   # noqa: BLE001 — nunca un traceback
        print("✗  No pude reconstruir la calificación completa: no coincide con datos/pedidos.csv "
              f"({error}). Vuelve a ejecutar la celda que crea los datos y [cargar], sin cambiarlas")
        return None


# ---------------------------------------------------------------------------------------------
# Auxiliares de los verificadores
# ---------------------------------------------------------------------------------------------

def _no_es_tabla(nombre, x, agrupado="todavía sin la muestra: falta .sample(...) al final"):
    """Mensaje para lo que no es un DataFrame (o None si sí lo es)."""
    import numbers
    import numpy as np
    import pandas as pd

    if isinstance(x, pd.DataFrame):
        return None
    if type(x).__name__.endswith("GroupBy"):
        return f"{nombre}: es el agrupamiento, {agrupado}"
    if isinstance(x, pd.Series) and len(x) and pd.api.types.is_bool_dtype(x):
        return (f"{nombre}: es la condición, un True o False por pedido; úsala para quedarte con "
                "los pedidos: pedidos[condición]")
    if isinstance(x, pd.Series) and x.name in ("count", "proportion"):
        return (f"{nombre}: es un conteo (value_counts); hace falta la muestra misma, la tabla con "
                "sus pedidos")
    if isinstance(x, pd.Series):
        return (f"{nombre}: es una sola columna (una serie); hace falta la tabla de pedidos con "
                "todas sus columnas: ¿elegiste una columna con [\"...\"]?")
    if x is None:
        return f"{nombre}: está vacía (None): ¿le asignaste el resultado de un print?"
    if isinstance(x, str):
        return f"{nombre}: es texto; hace falta la tabla de pedidos que devuelve pandas"
    if callable(x):
        return f"{nombre}: es una función, no su resultado: ¿faltaron los paréntesis del final, ()?"
    if isinstance(x, (bool, np.bool_)):
        return f"{nombre}: es True o False; hace falta una tabla de pedidos"
    if isinstance(x, (numbers.Number, np.number)):
        return (f"{nombre}: es un solo número; hace falta la tabla de pedidos de la muestra "
                "(un DataFrame), no cuántos son")
    if isinstance(x, (list, tuple, dict, set, np.ndarray)):
        return f"{nombre}: no es una tabla de pandas: saca la muestra de pedidos con pandas"
    return f"{nombre}: tiene que ser una tabla de pandas (es {type(x).__name__})"


def _filas(nombre, x, original, repetidos="¿usaste replace=True?"):
    """Los id_pedido de las filas de x, comprobando que son pedidos reales y sin cambios.

    Usa la columna id_pedido, el índice si son los id_pedido (set_index) o, si no, el número de
    fila original (el índice). Devuelve (lista de id, None) o (None, mensaje)."""
    import numpy as np
    import pandas as pd

    indice = x.index.get_level_values(-1) if isinstance(x.index, pd.MultiIndex) else x.index
    if {"count", "mean", "std"} <= set(map(str, x.index)):
        return None, (f"{nombre}: describe() resume la tabla en unas medidas; hace falta la muestra "
                      "misma, con sus pedidos")
    if "id_pedido" in x.columns:
        ids = x["id_pedido"]
        if isinstance(ids, pd.DataFrame):
            return None, f"{nombre}: tiene la columna id_pedido repetida"
        ids = ids.astype(str).tolist()
        por_id = original.set_index("id_pedido")
        if not set(ids) <= set(por_id.index):
            return None, (f"{nombre}: trae filas que no son pedidos de la tabla: saca la muestra "
                          "de pedidos sin cambiarla")
        filas = por_id.loc[ids]
        posiciones = None
    elif len(indice) and pd.Index(indice).astype(str).isin(original["id_pedido"]).all():
        ids = pd.Index(indice).astype(str).tolist()          # set_index("id_pedido")
        filas = original.set_index("id_pedido").loc[ids]
        posiciones = None
    elif pd.api.types.is_integer_dtype(indice) and len(indice) and \
            indice.min() >= 0 and indice.max() < len(original):
        posiciones = np.asarray(indice)
        filas = original.iloc[posiciones]
        ids = filas["id_pedido"].astype(str).tolist()
    else:
        return None, (f"{nombre}: no encuentro el código de cada pedido (id_pedido) ni su número de "
                      "fila: ¿quitaste columnas o cambiaste el índice? Saca la muestra de pedidos "
                      "tal como está")
    if len(set(ids)) < len(ids):
        return None, (f"{nombre}: hay pedidos repetidos: {repetidos} En una muestra cada "
                      "pedido entra una sola vez")
    cambiados = (f"{nombre}: hay pedidos con valores cambiados (en {{}}): saca la muestra de "
                 "pedidos sin modificarla")
    if posiciones is not None:   # sin id_pedido, el índice tiene que ser el número de fila original
        cambiados = (f"{nombre}: no sé de qué pedido es cada fila: ¿reiniciaste el índice "
                     "(reset_index) o quitaste la columna id_pedido? Deja la muestra como sale de "
                     "pandas")
    for c in ["tiempo_entrega_min", "distancia_km", "valor_pedido"]:
        if c in x.columns and not isinstance(x[c], pd.DataFrame):
            try:
                propios = pd.to_numeric(x[c], errors="raise").to_numpy(dtype=float)
            except (TypeError, ValueError):
                return None, f"{nombre}: la columna {c} ya no tiene números: ¿la cambiaste?"
            if not np.allclose(propios, filas[c].to_numpy(dtype=float), equal_nan=True):
                return None, cambiados.format(c)
    if "ciudad" in x.columns and not isinstance(x["ciudad"], pd.DataFrame):
        if list(x["ciudad"].astype(str)) != list(filas["ciudad"].astype(str)):
            return None, cambiados.format("ciudad")
    return ids, None


def _sin_sorteo(original, ids, grupo="ciudad"):
    """¿Las filas de cada grupo siguen un patrón fijo, sin sorteo?

    "seguidas" si en cada grupo son filas consecutivas (head, tail o un tramo); "desde_primera" si
    van a paso fijo desde la primera fila de cada grupo (iloc[::k]); "paso_azar" si van a paso fijo
    con otro inicio (un sistemático con inicio al azar, que sí es probabilístico); None si no hay
    patrón (hubo sorteo). Con las semillas de 100 a 999, ninguna muestra al azar tiene patrón."""
    import numpy as np

    orden = original.groupby(grupo).cumcount().set_axis(original["id_pedido"].astype(str))
    g = original.set_index("id_pedido").loc[ids, grupo]
    pasos, inicios = set(), set()
    for _, q in orden.loc[ids].groupby(g.values):
        q = np.sort(q.to_numpy())
        if len(q) < 3 or len(set(np.diff(q))) > 1:
            return None
        pasos |= set(np.diff(q).tolist())
        inicios.add(int(q[0]))
    if pasos == {1}:
        return "seguidas"
    return "desde_primera" if inicios == {0} else "paso_azar"


def _dias_sin_sorteo(original, fechas):
    """Lo mismo para los días: "seguidas", "desde_primera", "paso_azar" o None."""
    import numpy as np

    todas = sorted(original["fecha"].astype(str).unique())
    pos = sorted(todas.index(f) for f in fechas)
    pasos = set(np.diff(pos).tolist())
    if len(pos) < 3 or len(pasos) > 1:
        return None
    if pasos == {1}:
        return "seguidas"
    return "desde_primera" if pos[0] == 0 else "paso_azar"


# ---------------------------------------------------------------------------------------------
# Ejercicio 1 · Una muestra estratificada proporcional
# ---------------------------------------------------------------------------------------------

@_seguro
def revisar_estratificada(muestra=..., *sobran, **otros):
    """Ejercicio 1: muestra estratificada por ciudad con el 2 % de cada ciudad."""
    import pandas as pd

    if _sobrantes(sobran, otros, ["muestra"]):
        return
    if muestra is Ellipsis:
        print("·  muestra: todavía tiene ...")
        return
    problema = _no_es_tabla("muestra", muestra)
    if problema:
        print("✗  " + problema)
        return
    original = _original()
    if original is None:
        return
    n = len(original)
    x = muestra
    if len(x) == 0:
        print("✗  muestra: está vacía: ¿filtraste los pedidos antes de sacarla? La muestra sale de "
              "pedidos, agrupando por ciudad")
        return
    ids, problema = _filas("muestra", x, original)
    if problema:
        print("✗  " + problema)
        return
    if len(ids) == n:
        print("✗  muestra: es la tabla entera: falta sacar la muestra (el 2 % de cada ciudad)")
        return
    poblacion = original["ciudad"].value_counts()
    ciudad = original.set_index("id_pedido").loc[ids, "ciudad"]   # la ciudad de cada pedido
    obtenido = ciudad.value_counts().reindex(poblacion.index, fill_value=0)
    esperado = (poblacion * 0.02).round().astype(int)
    faltan = [c for c in poblacion.index if obtenido[c] == 0]

    def proporcional(fraccion):
        return bool(((obtenido - (poblacion * fraccion).round()).abs() <= 1).all())

    def agrupo_estrato():   # la confusión que avisa la diapositiva: la columna estrato
        por_estrato = original["estrato"].value_counts()
        en_muestra = original.set_index("id_pedido").loc[ids, "estrato"].value_counts()
        en_muestra = en_muestra.reindex(por_estrato.index, fill_value=0)
        return bool((en_muestra == (por_estrato * 0.02).round()).all())

    if proporcional(0.02):
        patron = _sin_sorteo(original, ids)
        if patron == "seguidas":
            print("✗  muestra: en cada ciudad tomaste pedidos seguidos (los primeros, los últimos o un "
                  "tramo): no hay sorteo, es una muestra por conveniencia. Agrupa por ciudad y usa "
                  ".sample(frac=0.02, random_state=mi_semilla)")
            return
        if patron == "desde_primera":
            print("✗  muestra: en cada ciudad tomaste uno de cada tantos desde el primero: no hay "
                  "sorteo, es una muestra por conveniencia. Agrupa por ciudad y usa "
                  ".sample(frac=0.02, random_state=mi_semilla)")
            return
        print("✓  muestra: el 2 % de cada ciudad")
        if patron == "paso_azar":
            print("·  Es un sistemático dentro de cada ciudad, con inicio al azar: también es "
                  "probabilístico. El ejercicio pide el estratificado de la celda [estratificado], "
                  "con .sample")
        sin_cambiar = original.groupby("ciudad").sample(frac=0.02, random_state=123)
        if sorted(ids) == sorted(sin_cambiar["id_pedido"].astype(str)):
            print("·  Es la muestra de mi_semilla = 123: cámbiala por un número de tres cifras "
                  "que elijas tú y vuelve a ejecutar")
        tabla = pd.DataFrame({"pedidos": poblacion, "en tu muestra": obtenido,
                              "% de la ciudad": (obtenido / poblacion * 100).round(2)})
        tabla.index.name = "ciudad"
        _mostrar(tabla)
        tiempo = original.set_index("id_pedido").loc[ids, "tiempo_entrega_min"]
        print(f"Tu x̄ = {tiempo.mean():.2f} min · μ = {original['tiempo_entrega_min'].mean():.2f} "
              "min (los 12 000 pedidos)")
        print("✓  Cuadra. Responde abajo: con una aleatoria simple de 240 pedidos, ¿saldrían siempre "
              "24 de Bucaramanga? ¿Por qué tu estratificada sí trae siempre el mismo número?")
        return
    if faltan and len(faltan) < len(poblacion):
        print(f"✗  muestra: le falta{'n' if len(faltan) > 1 else ''} {', '.join(faltan)}: ¿filtraste "
              "los pedidos? La muestra estratificada sortea en todas las ciudades")
    elif len(set(obtenido)) == 1 and len(poblacion) > 1:
        print("✗  muestra: todas las ciudades traen el mismo número de pedidos: eso es el mismo n "
              "por ciudad. Proporcional es la misma fracción en cada ciudad: usa frac=, no n=")
    elif proporcional(1 / 30):
        print("✗  muestra: esa es la fracción del ejemplo de la clase, 1 de cada 30; aquí va el 2 % "
              "(frac=0.02)")
    elif agrupo_estrato():
        print("✗  muestra: es proporcional por la columna estrato, no por ciudad: en muestreo, el "
              "estrato es el grupo que se sortea por separado, y aquí es la ciudad: groupby(\"ciudad\")")
    elif abs(len(ids) - esperado.sum()) <= 3:
        print("✗  muestra: el total está bien, pero las ciudades no traen su parte: ¿agrupaste por "
              "ciudad antes de .sample? Sin agrupar, cada ciudad sale con lo que toque al azar")
    elif original.set_index("id_pedido").loc[ids, "calificacion"].notna().all():
        print("✗  muestra: solo trae pedidos calificados: ¿filtraste (dropna) antes de sacarla? El 2 % "
              "es de todos los pedidos de cada ciudad")
    elif proporcional(len(ids) / n):
        parte = f"{len(ids) / n * 100:.1f}".replace(".", ",")
        print(f"✗  muestra: es proporcional, pero con el {parte} % de cada ciudad; se pide el 2 % "
              "(frac=0.02). Si ya usaste 0.02, ¿filtraste los pedidos antes de agrupar?")
    else:
        print("✗  muestra: no cuadra: agrupa por ciudad y saca el 2 % de cada una con "
              "frac=0.02 y tu semilla")


# ---------------------------------------------------------------------------------------------
# Ejercicio 2 · La insatisfacción de Bogotá, ponderada
# ---------------------------------------------------------------------------------------------

def _ponderar(d, mayor_igual=False, pesos=None):
    """Las cifras del 1 y 2 (y del 4 y 5) en %, sin ponderar y ponderadas, y sus errores típicos."""
    import numpy as np
    import pandas as pd

    marca = d["tiempo_entrega_min"] >= 55 if mayor_igual else d["tiempo_entrega_min"] > 55
    grupo = pd.Series(np.where(marca, "más de 55 min", "55 min o menos"), index=d.index)
    k = d["calificacion"].notna()
    peso = grupo.value_counts() / grupo[k].value_counts() if pesos is None else pesos
    w = grupo[k].map(peso).astype(float)
    estrellas = d.loc[k, "calificacion"].astype(int)
    y12, y45 = (estrellas <= 2).astype(int), (estrellas >= 4).astype(int)
    return {"sin": y12.mean() * 100, "pond": (w * y12).sum() / w.sum() * 100,
            "sin45": y45.mean() * 100, "pond45": (w * y45).sum() / w.sum() * 100,
            "den_todos": (w * y12).sum() / grupo.map(peso).astype(float).sum() * 100,
            "den_pedidos": y12.sum() / len(d) * 100,
            "invertidos": ((1 / w) * y12).sum() / (1 / w).sum() * 100,
            "entre_calificados": (w * y12).sum() / k.sum() * 100,
            "media": estrellas.mean(), "media_pond": (w * estrellas).sum() / w.sum(),
            "peso": peso}


@_seguro
def revisar_ponderada(*sueltos, sin_ponderar=..., ponderado=..., **otros):
    """Ejercicio 2: % de 1 o 2 estrellas en Bogotá, sin ponderar y ponderado por el tiempo."""
    import pandas as pd

    nombres = ["sin_ponderar", "ponderado"]
    if sueltos:
        print("✗  Pasa los valores por nombre, como en la plantilla: "
              "revisar_ponderada(sin_ponderar=sin_ponderar, ponderado=ponderado)")
        return
    if _sobrantes((), otros, nombres):
        return
    original = _original()
    if original is None:
        return
    bog = original[original["ciudad"] == "Bogotá"]
    b = _ponderar(bog)
    b_ge = _ponderar(bog, mayor_igual=True)
    t = _ponderar(original)
    t_ge = _ponderar(original, mayor_igual=True)
    b_nac = _ponderar(bog, pesos=t["peso"])
    verdad = {"sin_ponderar": [b["sin"]], "ponderado": [b["pond"], b_ge["pond"]]}
    tol, tol_pista = 0.06, 0.006
    # Las pistas, en el orden del diseño; cada una con las cifras que la delatan
    pistas = {
        "ponderado": [
            ([b["den_todos"]], "el denominador incluye a todos los pedidos: la suma va solo sobre "
                               "los calificados (filtra antes de sumar los pesos)"),
            ([t["pond"], t_ge["pond"], t["sin"]], "esa es la cifra de todos los pedidos, no la de "
                                                   "Bogotá: ¿cambiaste la línea datos =?"),
            ([b["den_pedidos"]], "el denominador son todos los pedidos de Bogotá; el % se calcula "
                                 "entre los calificados"),
            ([b["invertidos"]], "los pesos están al revés: el peso de cada grupo es pedidos ÷ "
                                "calificados"),
            ([b["entre_calificados"]], "dividiste entre el número de calificados; con pesos, se "
                                       "divide entre la suma de los pesos"),
            ([b["media"], b["media_pond"]], "esa es la media de las estrellas: la satisfacción no "
                                            "se promedia (¿usaste np.average?); cuenta los pedidos de 1 "
                                            "o 2 estrellas"),
            ([b["sin45"], b["pond45"]], "ese es el % de 4 o 5 estrellas; aquí va el de 1 o 2"),
            ([b_nac["pond"]], "usaste los pesos de todos los pedidos; Bogotá tiene los suyos: "
                              "calcúlalos dentro de datos"),
            ([b["sin"]], "ese es el % sin ponderar. Si los pesos salieron 1, datos solo trae los "
                         "calificados: tiene que traer todos los pedidos de Bogotá (el peso es "
                         "pedidos ÷ calificados). Si los pesos están bien, ¿multiplicaste cada "
                         "calificación por su peso?"),
        ],
        "sin_ponderar": [
            ([b["den_todos"]], "el denominador incluye a todos los pedidos; sin ponderar es el % "
                               "entre los calificados"),
            ([t["sin"], t["pond"]], "esa es la cifra de todos los pedidos, no la de Bogotá: "
                                    "¿cambiaste la línea datos =?"),
            ([b["den_pedidos"]], "el denominador son todos los pedidos de Bogotá; el % se calcula "
                                 "entre los calificados"),
            ([b["media"], b["media_pond"]], "esa es la media de las estrellas: la satisfacción no "
                                            "se promedia; cuenta los pedidos de 1 o 2 estrellas"),
            ([b["sin45"], b["pond45"]], "ese es el % de 4 o 5 estrellas; aquí va el de 1 o 2"),
            ([b["pond"]], "ese es el ponderado; sin ponderar es el % entre los calificados, sin "
                          "pesos"),
        ],
    }

    def _es(x, valores):
        if x is Ellipsis:
            return False
        val = _un_numero(x, len(original))[0]
        return any(_cerca(val, v, tol) or _cerca(val, v / 100, tol / 100) for v in valores)

    if _es(sin_ponderar, verdad["ponderado"]) and _es(ponderado, verdad["sin_ponderar"]):
        print("✗  ¿Intercambiaste los dos? sin_ponderar va sin pesos y ponderado, con pesos")
        return
    bien = 0
    proporcion = False
    for nombre, x in [("sin_ponderar", sin_ponderar), ("ponderado", ponderado)]:
        if x is Ellipsis:
            print(f"·  {nombre}: todavía tiene ...")
            continue
        val, problema = _un_numero(x, len(original))
        if problema:
            if isinstance(x, (pd.Series, pd.DataFrame)) and len(x) in (
                    len(bog), int(bog["calificacion"].notna().sum()), len(original)):
                problema = "esa es la columna entera, un valor por pedido: falta resumirla"
            elif problema.startswith("salió nan"):
                problema = ("salió nan, que es lo que da 0 ÷ 0: ¿datos quedó sin pedidos? Escribe la "
                            "ciudad igual que en la tabla: \"Bogotá\", con tilde")
            print(f"✗  {nombre}: {problema}")
            continue
        if any(_cerca(val, v, tol) for v in verdad[nombre]):
            print(f"✓  {nombre}")
            bien += 1
            continue
        if any(_cerca(val, v / 100, tol / 100) for v in verdad[nombre]):
            print(f"✓  {nombre} (como proporción)")
            proporcion = True
            bien += 1
            continue
        for cifras, pista in pistas[nombre]:
            if any(_cerca(val, c, tol_pista) or _cerca(val, c / 100, tol_pista / 100)
                   for c in cifras):
                print(f"✗  {nombre}: {pista}")
                break
        else:
            if val < 0 or val > 100:
                print(f"✗  {nombre}: un porcentaje va de 0 a 100: revisa el cálculo")
            else:
                print(f"✗  {nombre}: no cuadra: revisa que datos sean los pedidos de Bogotá y que el "
                      "% sea de 1 o 2 estrellas entre los calificados")
    if proporcion:
        print("·  Diste una proporción (entre 0 y 1); el ejercicio lo pide en %: multiplica por 100")
    if bien == 2:
        tabla = pd.DataFrame({"sin ponderar": [b["sin"], t["sin"]],
                              "ponderado": [b["pond"], t["pond"]]},
                             index=["Bogotá", "Todos los pedidos"])
        tabla.index.name = "1 o 2 estrellas (%)"
        _mostrar(tabla.round(2))
        print("✓  Las dos cuadran. Responde abajo: en puntos, ¿por qué la corrección de Bogotá es "
              "mayor que la del total? En proporción, las dos se parecen")


# ---------------------------------------------------------------------------------------------
# Ejercicio 3 (opcional) · Una muestra por conglomerados
# ---------------------------------------------------------------------------------------------

@_seguro
def revisar_conglomerados(muestra=..., *sobran, **otros):
    """Ejercicio 3: todos los pedidos de exactamente 6 días (sin suponer cuántos pedidos son)."""
    import numpy as np
    import pandas as pd

    if _sobrantes(sobran, otros, ["muestra"]):
        return
    if muestra is Ellipsis:
        print("·  muestra: todavía tiene ...")
        return
    original = _original()
    if original is None:
        return
    n = len(original)
    por_dia = original["fecha"].value_counts()
    x = muestra
    if isinstance(x, pd.Series) and 0 < len(x) < n and \
            set(x.astype(str)) <= set(por_dia.index):
        print("✗  muestra: esos son los días (o sus fechas); falta quedarte con todos los pedidos "
              "de esos días")
        return
    if isinstance(x, (list, tuple, set, np.ndarray, pd.Index)) and getattr(x, "ndim", 1) == 1 \
            and 0 < len(x) < n and set(pd.Series(list(x)).astype(str)) <= set(por_dia.index):
        print("✗  muestra: esos son los días; falta quedarte con todos los pedidos de esos días")
        return
    problema = _no_es_tabla("muestra", x, agrupado="y en conglomerados no se agrupa: elige 6 días "
                                                 "al azar y quédate con sus pedidos con isin")
    if problema:
        print("✗  " + problema)
        return
    if len(x) == 0:
        print("✗  muestra: está vacía: ¿a isin le pasaste una tabla? isin necesita los días como una "
              "serie o una lista de fechas, no un DataFrame (y en el mismo formato que la columna "
              "fecha)")
        return
    if set(map(str, x.columns)) == {"fecha"}:
        print("✗  muestra: solo trae las fechas: esos son los días; falta quedarte con todos los "
              "pedidos de esos días")
        return
    ids, problema = _filas("muestra", x, original, repetidos="¿repetiste un día o usaste "
                                                           "replace=True?")
    if problema:
        print("✗  " + problema)
        return
    if len(ids) == n:
        print("✗  muestra: es la tabla entera: falta quedarte solo con los pedidos de 6 días")
        return
    fechas = original.set_index("id_pedido").loc[ids, "fecha"].astype(str)
    cuenta = fechas.value_counts()
    k = len(cuenta)
    completos = bool((cuenta == por_dia.reindex(cuenta.index)).all())
    if len(ids) == 6:
        print("✗  muestra: tiene 6 filas: son 6 pedidos, no 6 días. Un conglomerado entra entero: "
              "elige 6 días y quédate con todos sus pedidos")
    elif len(ids) == len(por_dia) and k == len(por_dia):
        print("✗  muestra: tiene una fila por día: eso es estratificar por día (un pedido de cada "
              "día). Aquí se sortean 6 días y entran todos sus pedidos")
    elif k == 6 and completos and _dias_sin_sorteo(original, cuenta.index) in ("seguidas",
                                                                               "desde_primera"):
        print("✗  muestra: esos 6 días van seguidos (o a paso fijo desde el primero): no los "
              "sorteaste, es una muestra por conveniencia. Sortéalos con .sample(n=6, "
              "random_state=mi_semilla)")
    elif k == 6 and completos:
        print(f"✓  muestra: todos los pedidos de 6 días ({len(ids)} pedidos)")
        if _dias_sin_sorteo(original, cuenta.index) == "paso_azar":
            print("·  Tus días van a paso fijo desde un inicio que no es el primero: si lo sorteaste, "
                  "es un sistemático de días, también probabilístico. El ejercicio pide sortear "
                  "los 6 con .sample")
        dias_123 = original["fecha"].drop_duplicates().sample(n=6, random_state=123)
        dias_clase = original["fecha"].drop_duplicates().sample(n=6, random_state=1)
        if set(cuenta.index) == set(dias_123.astype(str)):
            print("·  Son los días de mi_semilla = 123: cámbiala por un número de tres cifras que "
                  "elijas tú y vuelve a ejecutar")
        elif set(cuenta.index) == set(dias_clase.astype(str)):
            print("·  Son los días de la clase (semilla = 1): usa mi_semilla, con un número de tres "
                  "cifras que elijas tú, y vuelve a ejecutar")
        tabla = cuenta.sort_index().rename("pedidos").to_frame()
        tabla.index.name = "fecha"
        _mostrar(tabla)
        tiempo = original.set_index("id_pedido").loc[ids, "tiempo_entrega_min"]
        mu = original["tiempo_entrega_min"].mean()
        conveniencia = original[original["ciudad"] == "Bogotá"].head(400)["tiempo_entrega_min"].mean()
        lejos = f"{abs(tiempo.mean() - mu):.2f}".replace(".", ",")
        lejos_conv = f"{conveniencia - mu:.2f}".replace(".", ",")
        print(f"Tu x̄ = {tiempo.mean():.2f} min · μ = {mu:.2f} min (los 12 000 pedidos): quedó a "
              f"{lejos} min de μ")
        print(f"✓  Cuadra. Responde abajo: tu x̄ quedó a {lejos} min de μ, y la de conveniencia a "
              f"{lejos_conv}: ¿cuál diferencia es error muestral y cuál es sesgo? ¿Cuál se achicaría "
              "con una muestra más grande?")
    elif k == 6:
        print("✗  muestra: trae 6 días, pero no todos sus pedidos: un conglomerado entra entero. "
              "Quédate con todos los pedidos de esos días (pista: isin)")
    elif completos:
        print(f"✗  muestra: trae los pedidos completos de {k} día{'s' if k > 1 else ''}; se piden "
              "exactamente 6 días")
        if k < 6:
            print("·  Si sorteaste filas de pedidos, un día puede salir dos veces: sortea entre las "
                  "fechas sin repetir, pedidos[\"fecha\"].drop_duplicates()")
    else:
        print(f"✗  muestra: trae pedidos de {k} día{'s' if k > 1 else ''}, y no todos los de cada "
              "uno: elige 6 días al azar y quédate con todos sus pedidos")

# Crea datos/pedidos.csv: los mismos 12 000 pedidos de las sesiones 1 a 4
ruta = crear_datos("datos")

## 1 · Los datos, con sus escalas

Cargamos los pedidos y declaramos las escalas, como en las sesiones anteriores. Hoy también usamos
`numpy`, que se importa como `np`.

In [ ]:
# [cargar]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pedidos = pd.read_csv("datos/pedidos.csv")
# Las escalas que declaramos en la sesión 1
franjas = ["Mañana", "Almuerzo", "Tarde", "Noche"]
pedidos["franja"] = pd.Categorical(
    pedidos["franja"], categories=franjas, ordered=True)
pedidos["estrato"] = pd.Categorical(
    pedidos["estrato"], categories=[1, 2, 3, 4, 5, 6], ordered=True)
pedidos["calificacion"] = pd.Categorical(
    pedidos["calificacion"], categories=[1, 2, 3, 4, 5], ordered=True)
pedidos.shape

### Diccionario de datos (el mismo de la sesión 1)

| Columna | Qué registra | Valores o unidad |
|---|---|---|
| `id_pedido` | Código del pedido | P00001, P00002… |
| `fecha` | Día del pedido | del 1 de enero al 30 de junio de 2026 |
| `franja` | Momento del día en que se pidió | Mañana, Almuerzo, Tarde, Noche |
| `ciudad` | Ciudad de entrega | Bogotá, Medellín, Cali, Barranquilla, Bucaramanga |
| `categoria` | Tipo de comercio que despacha | Restaurante, Mercado, Farmacia, Mascotas, Floristería |
| `n_productos` | Unidades pedidas | conteo |
| `valor_pedido` | Valor de los productos, sin domicilio ni propina | pesos (COP) |
| `medio_pago` | Cómo se pagó | Efectivo, Tarjeta, Billetera digital |
| `estrato` | Estrato de la **dirección de entrega** (lo asigna el municipio a la vivienda) | 1 a 6 |
| `suscriptor` | Si quien pide paga la suscripción de Tingua | Sí / No |
| `distancia_km` | Distancia por vía del comercio a la dirección | kilómetros; la app registra hasta 14: una ruta más larga queda como 14 |
| `lluvia` | Si llovía en la zona durante la entrega | Sí / No |
| `temperatura_c` | Temperatura en la zona al momento de la entrega | grados Celsius |
| `tiempo_entrega_min` | Minutos desde que se confirma el pedido hasta que llega | minutos |
| `calificacion` | Estrellas que dio quien pidió; vacía si no calificó | 1 a 5 |

## La pregunta de hoy: ¿qué tan satisfecha está la clientela?

La lista mínima del tablero pide el % de 4 o 5 estrellas: 79,1 %. Pero ¿79 % **de quiénes**? De los
7099 pedidos que se calificaron, el 59,2 % de los 12 000; el 40,8 % no calificó. Las calificaciones
son una muestra que nadie diseñó: **se eligió sola**. Primero veremos cómo se diseña una muestra al
azar; después, cómo corregir una que se eligió sola. Antes de la cifra, pregunta quién respondió.

## 2 · Si la gerencia llamara a 400 pedidos, ¿a cuáles?

La gerencia quiere llamar a 400 pedidos para preguntar por su satisfacción. Antes de elegir:

- El **marco** es la lista de la que se sortea: aquí, `pedidos.csv`. Solo trae los pedidos
  **entregados**: un pedido cancelado nunca podría salir en la muestra. Es un problema de **cobertura**.
- La **unidad** es el pedido, no el cliente: la tabla no dice quién pidió.
- Comparamos los métodos con el **tiempo de entrega** porque lo conocemos para todos los pedidos: su
  media $\mu$ es el blanco al que debería apuntar cada muestra.

En un muestreo **probabilístico**, cada pedido tiene una probabilidad conocida y mayor que cero de
entrar. Eso es lo que permite inferir: la media de la muestra, $\bar x$, estima $\mu$. Cuánto puede
alejarse de $\mu$ se mide en la sesión 6.

### Aleatorio simple

Cualquier grupo de 400 pedidos tiene la misma probabilidad de salir. La **semilla** hace que la
muestra se pueda repetir: con la misma semilla sale la misma muestra.

In [ ]:
# [mas]
semilla = 1                     # la de la clase; cámbiala y compara
mu = pedidos["tiempo_entrega_min"].mean()   # el parámetro: lo conocemos
mas = pedidos.sample(n=400, random_state=semilla)
x_mas = mas["tiempo_entrega_min"].mean()
print(f"μ = {mu:.2f} min · x̄ = {x_mas:.2f} min")
print("De Bucaramanga:", (mas["ciudad"] == "Bucaramanga").sum(), "pedidos")

### Sistemático

Se elige un inicio al azar entre los primeros 30 pedidos y, desde ahí, uno de cada 30
($12\,000 / 400 = 30$). Funciona si el orden del archivo no tiene un ciclo que coincida con el paso.

In [ ]:
# [sistematico]
paso = len(pedidos) // 400                # 12 000 / 400 = 30
inicio = np.random.default_rng(semilla).integers(paso)   # de 0 a 29
sistematica = pedidos.iloc[inicio::paso]  # uno de cada 30
x_sis = sistematica["tiempo_entrega_min"].mean()
print("Inicio:", inicio, "· pedidos:", len(sistematica))
print(f"x̄ = {x_sis:.2f} min")

### Estratificado

Se sortea **dentro de cada ciudad** la misma fracción, 1 de cada 30: cada ciudad entra con su parte.

**Aviso:** en muestreo, un **estrato** es cualquier grupo que se sortea por separado; aquí, la ciudad.
No es la columna `estrato`.

In [ ]:
# [estratificado]
estratificada = pedidos.groupby("ciudad").sample(
    frac=1/30, random_state=semilla)      # 1 de cada 30 en cada ciudad
print("Pedidos:", len(estratificada))
pd.DataFrame({"estratificada": estratificada["ciudad"].value_counts(),
              "aleatoria simple": mas["ciudad"].value_counts()})

### Por conglomerados

Se sortean **grupos enteros**: 6 días al azar, con todos sus pedidos. Sirve cuando hay una lista de
grupos (días, barrios, sucursales) y no una de unidades. El tamaño de la muestra depende de los días
que salgan. Si se sortean grupos y después unidades dentro de cada grupo, el muestreo es **multietapa**.

In [ ]:
# [conglomerados]
fechas = pedidos["fecha"].drop_duplicates()       # los 181 días
dias = fechas.sample(n=6, random_state=semilla)   # 6 días al azar
conglomerados = pedidos[pedidos["fecha"].isin(dias)]   # todos sus pedidos
print("Pedidos:", len(conglomerados))
conglomerados["fecha"].value_counts().sort_index()

### ¿Qué tan cerca de $\mu$ cae cada una?

Añadimos una muestra **por conveniencia**: los primeros 400 pedidos de Bogotá, los más a la mano.

In [ ]:
# [comparar]
# Por conveniencia: los primeros 400 pedidos de Bogotá
conveniencia = pedidos[pedidos["ciudad"] == "Bogotá"].head(400)
muestras = {"Aleatoria simple": mas, "Sistemática": sistematica,
            "Estratificada": estratificada,
            "Por conglomerados": conglomerados,
            "Por conveniencia": conveniencia}
comparacion = pd.DataFrame({
    "pedidos": {m: len(d) for m, d in muestras.items()},
    "media": {m: d["tiempo_entrega_min"].mean()
              for m, d in muestras.items()}})
print(f"μ = {mu:.2f} min")
comparacion.round(2)

In [ ]:
# [metodos_grafica]
colores = ["#F46201" if m == "Por conveniencia" else "#8A2E9E"
           for m in comparacion.index]
fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.scatter(comparacion["media"], comparacion.index, s=110, color=colores,
           zorder=3)
for m, x in comparacion["media"].items():   # el valor, del lado de afuera
    lado = 1 if x > mu else -1
    ax.annotate(f"{x:.2f}", (x, m), xytext=(10 * lado, 0),
                textcoords="offset points", va="center",
                ha="left" if lado > 0 else "right")
filas = len(comparacion)
ax.vlines(mu, -0.5, filas - 0.5, color="#3A0F47", linewidth=2,
          linestyle="--")                   # la línea de μ
ax.text(mu, -0.9, f"μ = {mu:.2f} min (los 12 000 pedidos)", ha="left")
ax.set_ylim(filas - 0.5, -1.3)              # en el orden de la tabla
ax.set_xlim(comparacion["media"].min() - 1, comparacion["media"].max() + 1)
ax.set_title("La muestra por conveniencia cae lejos de μ")
ax.set_xlabel("Media del tiempo de entrega (min)")
plt.show()

Una sola muestra por método no dice cuál es más **precisa**: para eso hay que sacar muchas muestras
de cada uno y ver cuánto varían. Es lo que haremos en la sesión 6.

### Los no probabilísticos

- **Por conveniencia:** los que están a la mano, como los primeros 400 pedidos de Bogotá.
- **Voluntario:** responde quien quiere. **La calificación de la app es una muestra voluntaria.**
- **Por cuotas:** se llenan cupos por grupo (tantos de cada ciudad), pero dentro de cada cupo se
  elige a mano, no al azar.

Sin probabilidades conocidas no hay forma de medir cuánto puede alejarse $\bar x$ de $\mu$. Y más datos
no lo arreglan: en 1936, la revista *Literary Digest* recibió más de dos millones de respuestas y se
equivocó de ganador en la elección presidencial de los Estados Unidos (Squire, 1988, *Public Opinion
Quarterly*).

## 3 · ¿Quién calificó? Sesgo y no respuesta

- El **error muestral** es azar: cambia de una muestra a otra y baja cuando crece $n$ (sesión 6).
- El **sesgo** es sistemático: empuja siempre hacia el mismo lado y **no** baja cuando crece $n$.

| Sesgo | Qué pasa |
|---|---|
| De **cobertura** | El marco deja fuera a una parte de la población |
| De **selección** o **autoselección** | Quien entra a la muestra no se parece al resto |
| De **no respuesta** | Quien no responde no se parece a quien sí |
| De **medición** | La pregunta o el instrumento miden mal |

La calificación mezcla dos: la app se la pide a **todos** los pedidos y solo algunos responden. Como
de todos sabemos el tiempo, la tratamos como **no respuesta**: veremos de qué depende y cómo
corregirla.

### ¿Quién califica?

`calificacion` está vacía en los pedidos que no se calificaron. Partimos el tiempo de entrega en
tramos con cortes explícitos y contamos qué parte de cada tramo se calificó.

In [ ]:
# [tasa_tiempo]
tramo = pd.cut(pedidos["tiempo_entrega_min"],
               bins=[0, 15, 25, 35, 45, 55, 65, 210],
               labels=["hasta 15", "15–25", "25–35", "35–45", "45–55",
                       "55–65", "más de 65"])
califico = pedidos["calificacion"].notna()       # True si calificó
tasa = califico.groupby(tramo, observed=True).mean() * 100
print(pd.DataFrame({"pedidos": tramo.value_counts(sort=False),
                    "% calificado": tasa.round(1)}))
fig, ax = plt.subplots()
etiquetas = tasa.index.astype(str)
ax.bar(etiquetas[:5], tasa.values[:5], color="#8A2E9E",
       label="55 min o menos")
ax.bar(etiquetas[5:], tasa.values[5:], color="#F46201",
       label="Más de 55 min")
ax.axhline(califico.mean() * 100, color="#3A0F47", linewidth=2,
           linestyle="--",
           label=f"Todos los pedidos: {califico.mean() * 100:.1f} %")
ax.set_ylim(0, 100)                        # las barras empiezan en cero
ax.legend(loc="upper left")
ax.set_title("Los pedidos de más de 55 min se califican más")
ax.set_xlabel("Tiempo de entrega (min)")
ax.set_ylabel("% de pedidos calificados")
plt.show()

¿Cambia la tasa de calificación con la ciudad, la suscripción o la lluvia?

In [ ]:
# [tasa_grupos]
tasa_grupos = pd.concat({
    columna: califico.groupby(pedidos[columna]).mean() * 100
    for columna in ["ciudad", "suscriptor", "lluvia"]})
tasa_grupos.round(1)

Por ciudad, suscripción o lluvia, la tasa va de 57,6 a 59,9 %: casi no cambia. De estas columnas,
ninguna la mueve como el tiempo.

### Tres tipos de faltantes

| Si la calificación falta… | Qué pasa | Qué se hace |
|---|---|---|
| **completamente al azar** | Solo se pierde $n$ | Nada: los calificados se parecen a todos |
| **según algo que vemos** (en Tingua, el tiempo) | Los calificados no se parecen a todos, pero sabemos en qué | Se **pondera** |
| **según lo que no vemos** (por ejemplo, que con el mismo tiempo califiquen más los pedidos de clientes muy molestos) | Ninguna columna lo revela | Hay que salir a preguntar: sortear algunos pedidos sin calificación y llamar hasta obtener su respuesta |

En inglés se llaman MCAR, MAR y MNAR. Ponderar por el tiempo funciona porque el tiempo se conoce
**también** para los pedidos sin calificación. La tasa de calificación muestra de qué variable
**observada** depende la no respuesta; que no dependa también de lo que no vemos es un **supuesto**
que en la vida real no se puede comprobar.

### Pocos pedidos, mucho peso en la insatisfacción (1 o 2 estrellas)

In [ ]:
# [quien_pesa]
mas_55 = pedidos["tiempo_entrega_min"] > 55
calificados = pedidos[califico]               # solo los que calificaron
baja = calificados["calificacion"] <= 2       # 1 o 2 estrellas
partes = pd.Series({
    "Pedidos": mas_55.mean() * 100,
    "Calificaciones": mas_55[califico].mean() * 100,
    "De 1 o 2 estrellas": mas_55[califico][baja].mean() * 100})
print(partes.round(1))
fig, ax = plt.subplots()
ax.bar(partes.index, partes.values, color="#F46201")
for i, v in enumerate(partes.values):         # el valor sobre cada barra
    ax.text(i, v + 2, f"{v:.1f} %", ha="center")
ax.set_ylim(0, 100)
ax.set_title("Más de 55 min: pocos pedidos, mucho peso en el 1 o 2")
ax.set_ylabel("% que son de más de 55 min")
plt.show()

Los de más de 55 min son el 3,5 % de los pedidos, el 4,8 % de las calificaciones y el 63,1 % de las de
1 o 2 estrellas. Dominan la insatisfacción sobre todo porque esperar más trae calificaciones bajas;
como además se califican más, pesan de más: ese exceso es el sesgo que la ponderación quita.

### Ponderar

Si la no respuesta depende **solo** del tiempo (y no de lo que no vemos), dentro de cada grupo de
tiempo los pedidos sin calificación se parecen a los calificados: ese es el supuesto. El **peso** de
cada grupo es pedidos entre calificados: a cuántos pedidos representa cada calificación. La
proporción ponderada es

$$\hat p_w = \frac{\sum w_i\, y_i}{\sum w_i}$$

La suma va **solo sobre los calificados**; $w_i$ es el peso del grupo del pedido $i$, y $y_i = 1$ si el
pedido tiene 1 o 2 estrellas (y 0 si no). Es la misma idea de un estratificado con distinta fracción
en cada estrato. Con estos pesos, los de más de 55 min vuelven a pesar lo que pesan entre los
pedidos: el 3,5 %, no el 4,8 % de las calificaciones.

El grupo se arma con `np.where`, un texto por pedido. La línea `datos = pedidos` va sola a
propósito: en el ejercicio 2 solo cambia esa.

In [ ]:
# [ponderar]
datos = pedidos              # ejercicio 2: solo cambia esta línea
grupo = pd.Series(np.where(datos["tiempo_entrega_min"] > 55,
                           "más de 55 min", "55 min o menos"),
                  index=datos.index)
calificado = datos["calificacion"].notna()
peso = grupo.value_counts() / grupo[calificado].value_counts()
print("Pesos:", peso.round(2).to_dict())
w = grupo[calificado].map(peso)       # el peso de cada calificado
estrellas = datos.loc[calificado, "calificacion"]
y = (estrellas <= 2).astype(int)      # 1 si tiene 1 o 2 estrellas
sin_ponderar = y.mean() * 100
ponderado = (w * y).sum() / w.sum() * 100
print(f"1 o 2 estrellas: {sin_ponderar:.2f} % → {ponderado:.2f} %")
y45 = (estrellas >= 4).astype(int)    # 1 si tiene 4 o 5 estrellas
print(f"4 o 5 estrellas: {y45.mean() * 100:.2f} % → "
      f"{(w * y45).sum() / w.sum() * 100:.2f} %")

### La comprobación, solo posible en un caso ficticio

Tingua es inventado: el generador sabe qué calificación habría dado **cada** pedido. La función
`calificacion_completa()`, que está en la celda oculta, la recupera: devuelve una serie indexada por
`id_pedido`, que se cruza con `map` por código.

**En la vida real esta celda no existe: no la uses en el tablero.**

In [ ]:
# [comprobacion]
completa = pedidos["id_pedido"].map(calificacion_completa())  # por código
print(f"1 o 2: {sin_ponderar:.2f} % sin ponderar · "
      f"{ponderado:.2f} % ponderado · "
      f"{(completa <= 2).mean() * 100:.2f} % de verdad")

En el 1 o 2, ponderar acercó la cifra a la verdad: de 4,39 a 3,66 %, con la verdad en 3,80 %. ¿Y en el
4 o 5?

In [ ]:
# [comprobacion_45]
print(f"4 o 5: {y45.mean() * 100:.2f} % sin ponderar · "
      f"{(w * y45).sum() / w.sum() * 100:.2f} % ponderado · "
      f"{(completa >= 4).mean() * 100:.2f} % de verdad")

Ponderar quita el sesgo que depende de lo que vemos, no el azar. En el 4 o 5, el azar de quién
calificó dentro del grupo de 55 min o menos empujaba hacia el lado contrario del sesgo y, por
suerte, casi lo compensaba en la cifra sin ponderar; al quitar el sesgo, quedó solo el azar.
Medir ese azar es el tema de la sesión 6.

### La respuesta a la gerencia

Estimamos que cerca de 8 de cada 10 pedidos tendrían 4 o 5 estrellas, si, dentro de cada grupo de
tiempo, los pedidos sin calificación se parecen a los calificados. La insatisfacción es menor de lo
que dicen las calificaciones: ponderada, baja de 4,4 % a 3,7 %.

## 4 · Para tu tablero

El indicador de satisfacción **no cambia**: el % de 4 o 5 estrellas entre los pedidos calificados,
mostrando cuántos se calificaron. Lo recomendable es mostrar la **tasa de calificación** de cada
semana y advertir en la nota sobre los datos que el % es solo de los calificados. Ponderar es
opcional. Esta celda calcula la tasa por semana ISO y marca las
semanas incompletas; no tiene verificador. Mira el rango que imprime: una semana con unos puntos
menos de tasa no es una tendencia. No conviertas ruido en noticia.

In [ ]:
# [semanas_calificacion]
fecha = pd.to_datetime(pedidos["fecha"])
semana = fecha.dt.isocalendar().week.rename("semana")   # lunes a domingo
por_semana = pd.DataFrame({
    "pedidos": pedidos.groupby(semana).size(),
    "calificados": califico.groupby(semana).sum(),
    "dias": fecha.groupby(semana).nunique()})
por_semana["% calificado"] = (por_semana["calificados"]
                              / por_semana["pedidos"] * 100).round(1)
por_semana["completa"] = por_semana["dias"] == 7
print("De", por_semana["% calificado"].min(), "a",
      por_semana["% calificado"].max(), "% según la semana")
por_semana

---
# Práctica guiada

Tres ejercicios; el tercero es **opcional** (o para la casa). Cada celda termina con un verificador
que dice qué está bien y, en lo demás, te da una pista sin darte la respuesta. Reemplaza cada `...`
y ejecuta; puedes repetir la celda las veces que quieras. Las respuestas escritas van al final, en
«Tus respuestas»: esas no las revisa ningún verificador.

## Ejercicio 1 · Una muestra estratificada proporcional

Saca una muestra estratificada por ciudad con el **2 %** de los pedidos de cada ciudad. Pista: la
celda [estratificado].

- `frac=` es una proporción: el 2 % se escribe `0.02`, no `2`.
- `n=` es un número de pedidos por grupo. No uses `frac=` y `n=` a la vez.
- Cambia `mi_semilla` por un número de tres cifras que elijas tú.

Después responde abajo: con una aleatoria simple de 240 pedidos (sin agrupar), ¿cuántos de
Bucaramanga esperarías? ¿Saldrían siempre 24? ¿Por qué tu estratificada sí trae siempre el mismo
número, con cualquier semilla?

In [ ]:
# [ej1]
mi_semilla = 123   # cámbiala por un número de tres cifras que elijas tú
# 2 % = 0.02: frac=0.02 saca el 2 % de cada grupo
muestra = ...      # agrupa por ciudad y saca el 2 % de cada una
revisar_estratificada(muestra)

## Ejercicio 2 · La insatisfacción de Bogotá, ponderada

Repite la ponderación solo con los pedidos de **Bogotá**, con sus propios pesos. La celda es la de
[ponderar]: cambia **solo** la línea `datos =`. Deja la última línea como está: entrega cada valor,
en %, con su nombre.

Después responde abajo: en **puntos**, la corrección de Bogotá casi duplica la del total; en
proporción, las dos se parecen (18 % y 17 %). ¿Por qué en puntos es mayor? Pista: ¿qué parte de los
pedidos de Bogotá pasa de 55 min, frente al 3,5 % del total? Y en una celda nueva escribe `w.sum()`:
¿por qué da justo el número de pedidos de Bogotá?

In [ ]:
# [ej2]
datos = pedidos       # ← cambia esta línea: solo los pedidos de Bogotá
grupo = pd.Series(np.where(datos["tiempo_entrega_min"] > 55,
                           "más de 55 min", "55 min o menos"),
                  index=datos.index)
calificado = datos["calificacion"].notna()
peso = grupo.value_counts() / grupo[calificado].value_counts()
print("Pesos:", peso.round(2).to_dict())
w = grupo[calificado].map(peso)       # el peso de cada calificado
estrellas = datos.loc[calificado, "calificacion"]
y = (estrellas <= 2).astype(int)      # 1 si tiene 1 o 2 estrellas
sin_ponderar = y.mean() * 100
ponderado = (w * y).sum() / w.sum() * 100
revisar_ponderada(sin_ponderar=sin_ponderar, ponderado=ponderado)

## Ejercicio 3 (opcional) · Una muestra por conglomerados

Elige 6 días al azar con tu semilla y quédate con **todos** sus pedidos. Pista: la celda
[conglomerados]. Deja la última línea como está.

Después responde abajo: el verificador dice a cuántos minutos de $\mu$ quedó tu $\bar x$; la muestra
por conveniencia quedó a 3,98. ¿Cuál de las dos diferencias es error muestral y cuál es sesgo? ¿Cuál
se achicaría con una muestra más grande?

In [ ]:
# [ej3]
mi_semilla = 123        # la tuya, de tres cifras
dias = ...              # 6 días al azar
muestra_dias = ...      # todos los pedidos de esos días
revisar_conglomerados(muestra_dias)

### Tus respuestas

*Doble clic para escribir aquí.*

1. Ejercicio 1: con una aleatoria simple de 240 esperaría unos … de Bucaramanga, pero …; mi
   estratificada trae siempre el mismo número porque …
2. Ejercicio 2: en puntos, la corrección de Bogotá es mayor porque …; en proporción …; `w.sum()` da …
   porque …
3. Ejercicio 3: mi $\bar x$ quedó a … min de $\mu$; es error muestral (o sesgo) porque …; la de
   conveniencia, a 3,98, es … porque …; con una muestra más grande se achicaría …
4. En la comprobación, ponderar alejó el 4 o 5 de la verdad (79,07 → 80,09 %, con la verdad en
   79,23 %). ¿Quiere decir que no había que ponderar? ¿Qué quita la ponderación y qué no? …
5. Autoevaluación. En otra app de domicilios, ¿qué tipo de faltante es cada uno, y qué harías?
   - Los pedidos que llegan con algo dañado casi nunca se califican; la app no registra daños: …
   - De noche se califica menos que de día: …
   - Una falla borró las calificaciones de los pedidos cuyo código termina en 7: …
   - Quien paga en efectivo califica menos que quien usa tarjeta: …